# Muon v5 structural audit

Audits and repacks the completed Muon training output. No base model, training, or competition submission is used.


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import struct
import zipfile

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
EXTRACT_DIR = WORKING_ROOT / "muon_v5_audit_extract"
OUTPUT_ZIP = WORKING_ROOT / "submission.zip"
EXPECTED = ["adapter_config.json", "adapter_model.safetensors"]


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safetensors_header(path):
    with open(path, "rb") as handle:
        raw_length = handle.read(8)
        if len(raw_length) != 8:
            raise RuntimeError("safetensors header length is missing")
        header_length = struct.unpack("<Q", raw_length)[0]
        if header_length <= 0 or header_length > 100 * 1024 * 1024:
            raise RuntimeError(f"invalid safetensors header length: {header_length}")
        header = json.loads(handle.read(header_length))
    tensor_keys = [key for key in header if key != "__metadata__"]
    if not tensor_keys:
        raise RuntimeError("safetensors contains no tensor entries")
    return header_length, len(tensor_keys)


candidates = sorted(INPUT_ROOT.rglob("submission.zip"))
print("submission_zip_candidates:", [str(path) for path in candidates])
if not candidates:
    raise FileNotFoundError("Muon v5 submission.zip was not mounted")
source_zip = next(
    (
        path
        for path in candidates
        if "muon" in str(path).lower() or "training-v5" in str(path).lower()
    ),
    candidates[0],
)
print("selected_source_zip:", source_zip)

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)
with zipfile.ZipFile(source_zip, "r") as archive:
    names = sorted(archive.namelist())
    print("source_zip_namelist:", names)
    if names != EXPECTED:
        raise RuntimeError(f"unexpected source zip root: {names}")
    archive.extractall(EXTRACT_DIR)

config_path = EXTRACT_DIR / "adapter_config.json"
model_path = EXTRACT_DIR / "adapter_model.safetensors"
config = json.loads(config_path.read_text(encoding="utf-8"))
rank = int(config.get("r", 0))
target_modules = config.get("target_modules")
if rank <= 0 or rank > 32:
    raise RuntimeError(f"invalid LoRA rank: {rank}")
if not target_modules:
    raise RuntimeError("target_modules is missing")
if model_path.stat().st_size < 100 * 1024 * 1024:
    raise RuntimeError(f"adapter model is unexpectedly small: {model_path.stat().st_size}")
header_length, tensor_count = safetensors_header(model_path)

if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()
shutil.copy2(source_zip, OUTPUT_ZIP)
with zipfile.ZipFile(OUTPUT_ZIP, "r") as archive:
    output_names = sorted(archive.namelist())
if output_names != EXPECTED:
    raise RuntimeError(f"unexpected output zip root: {output_names}")

report = {
    "candidate": "nemotron-s7-muon-full-training-v5-audited",
    "source_kernel": "muelsyse111/nemotron-s7-muon-full-training-v5",
    "source_zip_sha256": sha256_file(source_zip),
    "source_zip_size_bytes": source_zip.stat().st_size,
    "submission_zip_sha256": sha256_file(OUTPUT_ZIP),
    "submission_zip_size_bytes": OUTPUT_ZIP.stat().st_size,
    "zip_namelist": output_names,
    "rank": rank,
    "rank_lte_32": rank <= 32,
    "target_modules": target_modules,
    "safetensors_header_length": header_length,
    "safetensors_tensor_count": tensor_count,
    "safetensors_opened": True,
}
(WORKING_ROOT / "nemotron-s7-muon-v5-audit_report.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)
print(json.dumps(report, indent=2))
print("OK: /kaggle/working/submission.zip is ready.")
